# 06 — ONNX Export and Quantization

> Companion to **Week 11**, Part 5 of the lecture.

## Why ONNX?

ONNX is the **"PDF of machine learning"**. You write your model in PyTorch or
sklearn, then export it to ONNX, and after that *anyone* can run it — on a
phone, in a browser, in C++, in Java, on a microcontroller. They don't need
PyTorch installed.

In this notebook we will:

1. Train a sklearn MLP on digits.
2. Convert it to ONNX.
3. Run it with ONNX Runtime.
4. Quantize the ONNX file to INT8 and compare.


## Step 1 — Install ONNX libraries (Colab only)


In [ ]:
%pip install -q onnx onnxruntime skl2onnx


## Step 2 — Train a sklearn model


In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as ort
from onnxruntime.quantization import QuantType, quantize_dynamic

workdir = Path("/tmp/week11_onnx")
workdir.mkdir(exist_ok=True)

X, y = load_digits(return_X_y=True)
X = X.astype(np.float32)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

clf = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=60, random_state=42)
clf.fit(X_train, y_train)
print("sklearn accuracy:", clf.score(X_test, y_test))


## Step 3 — Convert to ONNX and quantize

This is the magic. Three lines convert a sklearn model to a portable ONNX
file. Two more lines quantize that file to INT8.


In [ ]:
initial_type = [("float_input", FloatTensorType([None, X_train.shape[1]]))]
onnx_model = convert_sklearn(clf, initial_types=initial_type)

fp32_path = workdir / "digits_fp32.onnx"
fp32_path.write_bytes(onnx_model.SerializeToString())

int8_path = workdir / "digits_int8.onnx"
quantize_dynamic(str(fp32_path), str(int8_path), weight_type=QuantType.QUInt8)

print("FP32 file:", fp32_path, f"({os.path.getsize(fp32_path)/1024:.1f} KB)")
print("INT8 file:", int8_path, f"({os.path.getsize(int8_path)/1024:.1f} KB)")


## Step 4 — Run both with ONNX Runtime

Notice: from this point on, **we never import sklearn or PyTorch again**.
Inference is happening through `onnxruntime` only.


In [ ]:
fp32_session = ort.InferenceSession(str(fp32_path))
int8_session = ort.InferenceSession(str(int8_path))
input_name = fp32_session.get_inputs()[0].name


def accuracy(session):
    preds = session.run(None, {input_name: X_test})[0]
    return float(np.mean(preds == y_test))


def benchmark(session, runs=200):
    start = time.perf_counter()
    for _ in range(runs):
        session.run(None, {input_name: X_test})
    return (time.perf_counter() - start) * 1000 / runs


pd.DataFrame(
    [
        {
            "model":      "onnx_fp32",
            "size_kb":    os.path.getsize(fp32_path) / 1024,
            "accuracy":   accuracy(fp32_session),
            "latency_ms": benchmark(fp32_session),
        },
        {
            "model":      "onnx_int8",
            "size_kb":    os.path.getsize(int8_path) / 1024,
            "accuracy":   accuracy(int8_session),
            "latency_ms": benchmark(int8_session),
        },
    ]
)


### What you should see

- The INT8 ONNX file is roughly 3-4x smaller than the FP32 ONNX file.
- Accuracy is nearly identical (often within 0.5 percentage points).
- Latency is the same or slightly faster.

> 🧠 **Take-away:** ONNX gives you portability *and* a clean place to apply
> optimizations. You can quantize, prune, or fuse layers without touching the
> original training code.

## 🧪 Your turn

Try `weight_type=QuantType.QInt8` instead of `QuantType.QUInt8` (signed instead
of unsigned). Does accuracy change? Does file size change?
